# Comparison of different log-normal distributions

This notebook serves two purposes. 

It first provides a comparison of different forms of the log-normal distribution to give some insights into normalisation, maxima, means and standard deviations. 

Second, this notebook also reproduces Figure 14 in [Graber et al. (2024)](https://arxiv.org/abs/2312.14848).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy import stats

from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import mlpoppyns.generator.maps.axes_scaling as axs
import utilities.plot_settings

from matplotlib import rc

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 40
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=MEDIUM_SIZE)  # fontsize of the figure title

## Different types

Defining main parameters for a random comparison.

In [ ]:
mu = 2.0
sigma = 0.6
n_samples = 10000

Direct implementation of normalised PDF for log in base 10.

In [ ]:
def PDF_log10_direct(mu, sigma, x):
    pdf = (
        np.log10(np.e)
        / (x * sigma * np.sqrt(2 * np.pi))
        * np.exp(-((np.log10(x) - mu) ** 2) / (2 * sigma**2))
    )

    return pdf

Drawing samples from a log-normal in base 10.

In [ ]:
x_log10_samples = 10 ** (np.random.normal(mu, sigma, n_samples))

Direct implementation of normalised PDF for natural log.

In [ ]:
def PDF_log_direct(mu, sigma, x):
    pdf = (
        1
        / (x * sigma * np.sqrt(2 * np.pi))
        * np.exp(-((np.log(x) - mu) ** 2) / (2 * sigma**2))
    )

    return pdf

Drawing samples from natural logarithm two ways.

In [ ]:
x_log_samples_np = np.random.lognormal(mu, sigma, n_samples)

x_log_samples_direct = np.e ** (np.random.normal(mu, sigma, n_samples))

Plot for comparison.

In [ ]:
x = np.logspace(-1, 4, 5000)
x_edges = np.logspace(-1, 4, 51)

In [ ]:
colors = ["#fde725", "dodgerblue", "#440154"]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.plot(
    x,
    PDF_log10_direct(mu, sigma, x),
    linestyle="--",
    linewidth=4,
    color="tab:blue",
    alpha=1,
    label=r"PDF $\log_{10}$",
)
ax.hist(
    x_log10_samples,
    bins=x_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls="-",
    density=True,
    label=r"sample $10^{\text{random.normal}}$",
)

ax.plot(
    x,
    PDF_log_direct(mu, sigma, x),
    linestyle="--",
    linewidth=4,
    color=colors[2],
    alpha=1,
    label=r"PDF $\log_{e}$",
)
ax.hist(
    x_log_samples_np,
    bins=x_edges,
    histtype="step",
    edgecolor="slateblue",
    lw=4,
    ls="-",
    density=True,
    label=r"sample random.lognormal",
)
ax.hist(
    x_log_samples_direct,
    bins=x_edges,
    histtype="step",
    edgecolor=colors[1],
    lw=4,
    ls="-",
    density=True,
    label=r"sample ${\rm e}^{\text{random.normal}}$",
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$N$")
plt.xscale("log")
ax.grid()
ax.legend(frameon=True, loc=1, fontsize=24)

plt.show()

The first two representations of the logarithmic distributions and the last three are identical, as expected.

The fluctuations on the left flank of the histograms are due to the fact that we set the density to true. To normalise the distributions, the value of each bin is obtained by dividing the initial counts by the total count across the entire histogram and the respective bin width, i.e., `density = counts / (sum(counts) * np.diff(bins))`. As a result, the integral over the entire histogram is one, but the width of the bins varies (and is smaller for smaller $x$). This causes variation in the lefthand side of the histograms.

## Maxima

The peaks of the distributions, when plotted in a normalised fashion, i.e., such that the area integrates to one when the $x$ axis is given in log, are not located at the mean $\mu = 2$. Instead the mode can be determined as the place where the derivatives of the PDFs vanish.

For the base 10 case the mode is $10^{\mu} {\rm e}^{-\sigma^2 \log_e(10)^2}$:

In [ ]:
10**mu * np.e ** (-(sigma**2) * np.log(10) ** 2)

For the natural logarithm it is ${\rm e}^{\mu - \sigma^2}$:

In [ ]:
np.e ** (mu - sigma**2)

We can compare this to the direct maxima of the PDFs:

In [ ]:
x[
    np.where(
        PDF_log10_direct(mu, sigma, x) == max(PDF_log10_direct(mu, sigma, x))
    )[0][0]
]

In [ ]:
x[
    np.where(
        PDF_log_direct(mu, sigma, x) == max(PDF_log_direct(mu, sigma, x))
    )[0][0]
]

## Most common x value

Histogram samples without normalising and plot with log scale for $x$ axis: maximum is located as $10^{\mu} = 100$ and shape looks Gaussian as expected. We can overlay the Gaussian in log and rescale the maximum to see that this has indeed the same mean and sigma.

Define Gaussian to plot things with log space on the $x$ axis

In [ ]:
def PDF_normal_direct(mu, sigma, x):
    pdf = (
        1
        / (sigma * np.sqrt(2 * np.pi))
        * np.exp(-((x - mu) ** 2) / (2 * sigma**2))
    )

    return pdf

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    x_log10_samples,
    bins=x_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls="-",
    density=False,
    align="mid",
    label=r"sample $10^{\mathcal{N}}$",
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$N$")
ax.set_xscale("log")
ax.set_xlim(1e-1, 1e4)
ax.grid()

ax2 = ax.twiny()

ax2.plot(
    np.log10(x),
    1000 * PDF_normal_direct(mu, sigma, np.log10(x)),
    linestyle="-",
    linewidth=5,
    color="tab:blue",
    alpha=1,
    label="\mathcal{N(\log_{10} x)}",
)
ax2.set_xlim(-1, 4)
ax2.set_xlabel(r"$\log_{10} x$")
ax.legend(frameon=True, loc=2)
ax.legend(frameon=True, loc=2)


plt.show()

Same holds in case we plot things with a linear $x$ axis and zoom in on the peak:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.hist(
    x_log10_samples,
    bins=x_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls="-",
    density=False,
    label=r"sample $10^{\mathcal{N}}$",
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$N$")
ax.set_xlim(0, 400)
ax.grid()
ax.legend(frameon=True, loc=0)

plt.show()

## Different mus and sigmas

For comparison: look at what the peak and width do for different parameters in the density picture.

In [ ]:
print(mu)
print(sigma)

In [ ]:
x_wide = np.logspace(-15, 10, 5000)

Rescaled $y$ axis to see the peaks equally well (note that means the curves are no longer normalised):

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.plot(
    x_wide,
    PDF_log10_direct(mu / 2, sigma, x),
    linestyle="-",
    linewidth=4,
    color="tab:blue",
    alpha=1,
    label=f"$\mu = {mu / 2}, \sigma = {sigma}$",
)
ax.plot(
    x_wide,
    10 * PDF_log10_direct(mu, sigma, x),
    linestyle="-.",
    linewidth=4,
    color="tab:blue",
    alpha=1,
    label=f"$\mu = {mu}, \sigma = {sigma}$",
)
ax.plot(
    x_wide,
    1000 * PDF_log10_direct(mu * 2.0, sigma, x),
    linestyle="--",
    linewidth=4,
    color="tab:blue",
    alpha=1,
    label=f"$\mu = {mu * 2.0}, \sigma = {sigma}$",
)
ax.plot(
    x_wide,
    PDF_log10_direct(mu / 2.0, sigma / 2.0, x),
    linestyle="-",
    linewidth=4,
    color="tab:red",
    alpha=1,
    label=f"$\mu = {mu/2.0}, \sigma = {sigma/2.0}$",
)
ax.plot(
    x_wide,
    PDF_log10_direct(mu / 2.0, 0.8, x),
    linestyle="-.",
    linewidth=4,
    color="tab:red",
    alpha=1,
    label=f"$\mu = {mu/2.0}, \sigma = {0.8}$",
)
ax.plot(
    x_wide,
    5 * PDF_log10_direct(mu, 0.8, x),
    linestyle="-",
    linewidth=4,
    color="seagreen",
    alpha=1,
    label=f"$\mu = {mu}, \sigma = {0.8}$",
)

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$N$")
plt.xscale("log")
ax.grid()
ax.legend(frameon=True, loc=0)

plt.show()

Observations: (1) for the same sigma, the peak moves to the right when mu increases; (2) for smaller sigma the width of the Gaussian decreases as expected but the peak moves (see definition of mode above); (3) as a result of this the peak of a curve with larger mu and wider sigma might be located to the left of something with a smaller mu.

## Paper comparison plot

Comparison between our best values and those in the literature.

In [ ]:
mu_B_Graber = 13.10
sigma_B_Graber = 0.45

mu_P_Graber = -1.00
sigma_P_Graber = 0.38

In [ ]:
# Model with 45 radio PSRs with SNRs attached only.
# No field decay for first 10^5 yrs then yes.

# See Table 1 in Igoshev et al (2022).
mu_B_Igoshev = 12.44
sigma_B_Igoshev = 0.44

# MCMC analysis for their model A.
# See Table 2 in Igoshev et al (2022).
mu_P_Igoshev = -1.04
sigma_P_Igoshev = 0.53

In [ ]:
# Model D for radio-pulsar population with light-element envelope and slow field decay.
# See Table 1 in Gullon et al. (2015).
mu_B_Gullon = 12.99
sigma_B_Gullon = 0.56

In [ ]:
# Results for the rotational model (although both are almost identical) with exponential field decay.
# See Table 2 in Cieslar et al. (2020).
mu_B_Cieslar = 12.67
sigma_B_Cieslar = 0.34

In [ ]:
# No field decay included.
# See Table 6 in Faucher-Giguere and Kaspi (2006).
mu_B_Faucher = 12.65
sigma_B_Faucher = 0.55

In [ ]:
x_comparison_Bfield = np.linspace(11, 16, 1000)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

ax.plot(
    x_comparison_Bfield,
    PDF_normal_direct(mu_B_Faucher, sigma_B_Faucher, x_comparison_Bfield),
    linestyle=(0, (1, 1)),
    linewidth=4,
    color=colors[2],
    alpha=1,
    label=r"FGK (2006)",
)
ax.plot(
    x_comparison_Bfield,
    PDF_normal_direct(mu_B_Gullon, sigma_B_Gullon, x_comparison_Bfield),
    linestyle="-.",
    linewidth=4,
    color="#21918c",
    alpha=1,
    label=r"Gullón et al. (2015)",
)
ax.plot(
    x_comparison_Bfield,
    PDF_normal_direct(mu_B_Cieslar, sigma_B_Cieslar, x_comparison_Bfield),
    linestyle=(5, (10, 3)),
    linewidth=4,
    color="darkblue",
    alpha=1,
    label=r"Cieślar et al. (2021)",
)
ax.plot(
    x_comparison_Bfield,
    PDF_normal_direct(mu_B_Igoshev, sigma_B_Igoshev, x_comparison_Bfield),
    linestyle="--",
    linewidth=4,
    color=colors[1],
    alpha=1,
    label=r"Igoshev et al. (2022)",
)
ax.plot(
    x_comparison_Bfield,
    PDF_normal_direct(mu_B_Graber, sigma_B_Graber, x_comparison_Bfield),
    linestyle="-",
    linewidth=6,
    color="black",
    alpha=1,
    label=r"This work",
)

ax.set_xlim(11, 16)
ax.set_xlabel(r"Initial magnetic field $\log_{10} B_0$ [G]")
ax.grid()
ax.legend(frameon=True, loc=1)

plt.tight_layout()
plt.savefig(
    f"../../paper_plots/graber_etal_2024/plots/comparison_B_lognormals.pdf",
    dpi=400,
    bbox_inches="tight",
)
plt.show()

In [ ]:
x_comparison_P = np.linspace(-3, 1.5, 1000)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

ax.plot(
    x_comparison_P,
    PDF_normal_direct(mu_P_Igoshev, sigma_P_Igoshev, x_comparison_P),
    linestyle="--",
    linewidth=4,
    color=colors[1],
    alpha=1,
    label=r"Igoshev et al. (2022)",
)
ax.plot(
    x_comparison_P,
    PDF_normal_direct(mu_P_Graber, sigma_P_Graber, x_comparison_P),
    linestyle="-",
    linewidth=6,
    color="black",
    alpha=1,
    label=r"This work",
)

ax.set_xlim(-3, 1.5)
ax.set_xlabel(r"Period $\log_{10} P_0$ [s]")
ax.grid()
ax.legend(frameon=True, loc=1)

plt.tight_layout()
plt.savefig(
    f"../../paper_plots/graber_etal_2024/plots/comparison_P_lognormals.pdf",
    dpi=400,
    bbox_inches="tight",
)
plt.show()